In [5]:
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.metrics import accuracy_score, make_scorer
import warnings
import re

from sklearn.multioutput import MultiOutputClassifier
from catboost import CatBoostClassifier

In [6]:
warnings.filterwarnings('ignore') 
ohe = OneHotEncoder(sparse_output=False)

In [ ]:
df['build_type'].unique()

array(['normal', 'compact', 'rgb', 'stealth', 'dual-mon', nan,
       'M-ATX Tower', 'ATX Tower', 'ATX Mid Tower', 'Full Tower',
       'Full Tower Dual Chamber', 'ATX Workstation',
       'Mid Tower Dual Chamber', 'Full Tower Workstation',
       'M-ATX Mid Tower', 'AM5 Mid Tower', 'AM5 Tower', 'AM5 Workstation',
       'Mini-ITX / Small Form Factor', 'Gaming / Workstation', 'Gaming',
       'Workstation', 'AI Workstation', 'Extreme Gaming',
       'Dev Workstation', 'Gaming / Streaming', 'Audio Workstation',
       'Research Station', 'Video Workstation', 'Production Station',
       'Simulation Station', 'Gaming Showcase', 'Streaming Studio',
       'VR Workstation', 'Extreme Workstation', 'Data Workstation',
       'Server Workstation', 'Flagship Gaming', 'VFX Suite',
       'Ultimate Rig', 'Flagship Ultra', 'Streaming', 'Hybrid',
       'Production', 'Research', 'Video Suite', 'VFX Workstation',
       'Showcase', 'Flagship', 'Custom'], dtype=object)

In [8]:
df = pd.read_excel("dataset/rigbuilder_main_dataset.xlsx")
df.columns = df.columns.str.strip()

In [5]:
cabinet_df = pd.read_excel("dataset/rigbuilder_cabinet.xlsx")
clr_df = pd.read_excel("dataset/rigbuilder_cpu_cooler.xlsx")
cpu_df = pd.read_excel("dataset/rigbuilder_cpu.xlsx")
mobo_df = pd.read_excel("dataset/rigbuilder_mobo.xlsx")
monitor_df = pd.read_excel("dataset/rigbuilder_monitor.xlsx")
psu_df = pd.read_excel("dataset/rigbuilder_psu.xlsx")
ram_df = pd.read_excel("dataset/rigbuilder_ram.xlsx")
storage_df = pd.read_excel("dataset/rigbuilder_storage.xlsx")
gpu_df = pd.read_excel("dataset/rigbuilder_gpu.xlsx")


In [6]:
# Remove extra spaces from all column names

cpu_df.columns = cpu_df.columns.str.strip()
gpu_df.columns = gpu_df.columns.str.strip()
mobo_df.columns = mobo_df.columns.str.strip()
clr_df.columns = clr_df.columns.str.strip()
psu_df.columns = psu_df.columns.str.strip()
storage_df.columns = storage_df.columns.str.strip()
ram_df.columns = ram_df.columns.str.strip()
cabinet_df.columns = cabinet_df.columns.str.strip()
monitor_df.columns = monitor_df.columns.str.strip()

In [7]:
x = df[['cpu', 'gpu', 'mobo', 'cpu_cooler', 'psu', 'storage', 'ram', 'cabinet', 'monitor']].iloc[0].tolist()


In [8]:
df.drop(1, inplace=True)
df.reset_index(drop=True, inplace=True)

In [9]:
df['upgrade_path'].replace({'small ': 'small'}, inplace=True)
df['budget'] = pd.to_numeric(df['budget'], errors='coerce')

In [10]:
df["used_parts"] = df["used_parts"].map({
    "yes": 1,
    "no": 0
})

df["monitor_required"] = df["monitor_required"].map({
    "yes": 1,
    "no": 0
})

df["monitor_size"] = (
    df["monitor_size"]
    .replace(["none", "no"], np.nan)
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
    .astype(float)
    .fillna(0)
)
# df["monitor_size"] = df["monitor_size"].astype(int)

In [11]:
cols = ['cpu', 'gpu', 'mobo', 'cpu_cooler', 'psu', 'storage',
       'ram', 'cabinet', 'monitor']

In [12]:
df = df.dropna(subset=cols)
df.reset_index(drop=True, inplace=True)
print("df:", df.shape)

df: (739, 19)


In [ ]:
X = df[['budget', 'used_parts', 'usage_scenario', 'colour_theme',
       'monitor_required', 'monitor_size', 'monitor_resolution', 'build_type',
       'upgrade_path']]

Y = df[['cpu', 'gpu', 'mobo', 'cpu_cooler', 'psu', 'storage',
       'ram', 'cabinet', 'monitor']]

In [14]:
categorical_cols = ["usage_scenario", "colour_theme", "monitor_resolution", "build_type", "upgrade_path"]
df[categorical_cols] = df[categorical_cols].astype(str)

In [15]:
encoded_data = ohe.fit_transform(df[categorical_cols])
encoded_df = pd.DataFrame(encoded_data, columns=ohe.get_feature_names_out(categorical_cols))

In [16]:
X = X.drop(columns=categorical_cols)
X = pd.concat([X, encoded_df], axis=1)

In [17]:
Y["gpu"] = Y["gpu"].astype(str)

In [18]:
target_encoders = {}
for col in Y.columns:
    label_encoder = LabelEncoder()
    Y[col] = label_encoder.fit_transform(Y[col])
    target_encoders[col] = label_encoder


In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)


## **Random Forest Model**

In [20]:
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'max_features': ['sqrt', 'log2'],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]}

rf = RandomForestClassifier(random_state=42)

search_best_pera = GridSearchCV(estimator=rf, param_grid=param_dist, 
                                cv=5, n_jobs=-1)

search_best_pera.fit(X_train, y_train)
random_forest_model = search_best_pera.best_estimator_

y_pred = random_forest_model.predict(X_test)

In [21]:
# 1. Individual component accuracy
for i, col in enumerate(Y.columns):
    acc = accuracy_score(y_test.iloc[:, i], y_pred[:, i])
    print(f"{col:12} : {acc:.2%}")

# 2. Overall component accuracy
component_accuracy = np.mean(y_test.values == y_pred)

print(f"Component Accuracy : {component_accuracy:.2%}")

cpu          : 29.05%
gpu          : 39.86%
mobo         : 22.30%
cpu_cooler   : 51.35%
psu          : 53.38%
storage      : 29.73%
ram          : 43.24%
cabinet      : 25.68%
monitor      : 37.84%
Component Accuracy : 36.94%


## **Cat Model**

In [22]:
# Base CatBoost
cat = CatBoostClassifier(
    verbose=0,
    random_state=42,
    allow_writing_files=False)

# Multi-output
multi_cat = MultiOutputClassifier(cat)


# Hyperparameter grid
param_grid = {
    "estimator__iterations": [100, 200],
    "estimator__depth": [4, 6, 8],
    "estimator__learning_rate": [0.01, 0.1],
    "estimator__l2_leaf_reg": [1, 3, 5]}

# Custom accuracy
def multioutput_accuracy(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_pred = np.squeeze(y_pred)
    return np.mean(y_true == y_pred)

custom_accuracy = make_scorer(multioutput_accuracy)

# GridSearch
cat_model = GridSearchCV(
    estimator=multi_cat,
    param_grid=param_grid,
    cv=5,
    scoring=custom_accuracy,
    n_jobs=-1,
    verbose=0,
    error_score="raise")


In [23]:
# Train
cat_model.fit(X_train, y_train)

print(f"Best CV Score: {cat_model.best_score_:.2%}")
cat_model = cat_model.best_estimator_

Best CV Score: 36.49%


In [24]:
joblib.dump(random_forest_model, "models/random_forest_model.pkl")
joblib.dump(cat_model, "models/cat_model.pkl")


['models/cat_model.pkl']

In [25]:
cat_model = joblib.load("models/cat_model.pkl")
Forest_model = joblib.load("models/random_forest_model.pkl")

## **Decode predictions**
prediction → decoded component IDs → actual CPU/GPU/Motherboard names ka flow banayenge.

In [26]:
# Component ID columns
components_id_col_name = {
    "CPU": "cpu_id",
    "GPU": "gpu id",
    "Mother Board": "model_id",
    "Cooler": "Cooler ID",
    "PSU": "PSU ID",
    "Storage": "storage_id",
    "RAM": "ram_id",
    "Cabinet": "Cabinet ID",
    "Monitor": "Monitor ID"}

# Target columns
target_columns = [
    "cpu",
    "gpu",
    "mobo",
    "cpu_cooler",
    "psu",
    "storage",
    "ram",
    "cabinet",
    "monitor"]

# Price columns
price_columns = {
    
    "GPU": "gpu price",
    "Mother Board": "price",
    "Cooler": "Price",
    "PSU": "Approx Price (INR)",
    "Storage": "price",
    "RAM": "approx_price",
    "Cabinet": "Price",
    "Monitor": "Price",
}

# Component DataFrames
component_dataframes = {
    "CPU": cpu_df,
    "GPU": gpu_df,
    "Mother Board": mobo_df,
    "Cooler": clr_df,
    "PSU": psu_df,
    "Storage": storage_df,
    "RAM": ram_df,
    "Cabinet": cabinet_df,
    "Monitor": monitor_df}

### **1. User input preprocessing**

In [27]:
def preprocess_user_input(
    budget,
    used_parts,
    usage_scenario,
    colour_theme,
    monitor_required,
    monitor_size,
    monitor_resolution,
    build_type,
    upgrade_path):

    user_input = pd.DataFrame([{
        "budget": budget,
        "used_parts": used_parts,
        "usage_scenario": usage_scenario,
        "colour_theme": colour_theme,
        "monitor_required": monitor_required,
        "monitor_size": monitor_size,
        "monitor_resolution": monitor_resolution,
        "build_type": build_type,
        "upgrade_path": upgrade_path}])

    encoded_data = ohe.transform(
        user_input[categorical_cols])

    encoded_df = pd.DataFrame(
        encoded_data,
        columns=ohe.get_feature_names_out(categorical_cols))

    user_input = user_input.drop(columns=categorical_cols)

    user_input = pd.concat([user_input, encoded_df], axis=1)

    return user_input

### **compatibility Text**

In [28]:

def get_compatible_chipsets(compatible_socket):
    if pd.isna(compatible_socket):
        return []

    return re.findall(r'\b[A-Z]\d{3}\b', str(compatible_socket))

def check_cpu_mobo(cpu, mobo):

    cpu_socket = str(
        cpu["socket_no"]).strip()

    mobo_socket = str(
        mobo["socket_no"]).strip()

    # Socket check
    if cpu_socket != mobo_socket:
        return False

    # Chipset check
    compatible_chipsets = get_compatible_chipsets(
        cpu["compatible_socket"])

    mobo_chipset = str(
        mobo["chipset"]).strip()

    return mobo_chipset in compatible_chipsets


def check_ram_mobo(ram, mobo):

    ram_generation = str(
        ram["ram_generation"]
    ).strip().upper()

    mobo_ram_type = str(
        mobo["ram_type"]
    ).strip().upper()

    ram_compatible_sockets = [
        socket.strip().upper()
        for socket in str(ram["mobo_socket_compatibility"]).split("/")]

    mobo_socket = str(mobo["socket_no"]).strip().upper()

    if ram_generation != mobo_ram_type:
        return False

    if mobo_socket not in ram_compatible_sockets:
        return False

    return True


def check_gpu_psu(gpu, psu):

    gpu_required = int(
        str(gpu["psu required"]).replace("W", "").strip())

    psu_wattage = float(psu["Wattage (W)"])

    return psu_wattage >= gpu_required


def check_gpu_cabinet(gpu, cabinet):

    gpu_length = float(
        str(gpu["gpu length"]).replace("mm", "").strip())

    max_gpu_length = float(
        str(cabinet["Max GPU Length"]).replace("mm", "").strip())

    return gpu_length <= max_gpu_length


def get_watt(value):

    return float(
        str(value)
        .replace("W", "")
        .strip())


def check_cpu_cooler(cpu, cooler):

    cpu_socket = str(cpu["socket_no"]).strip().upper()

    compatible_sockets = [
        socket.strip().upper()
        for socket in str(
            cooler["Compatible Socket"]).split(",")]

    cpu_tdp = get_watt(cpu["tdp_wattage"])

    cooler_tdp = get_watt(cooler["Max Rated TDP"])

    socket_ok = (cpu_socket in compatible_sockets)

    tdp_ok = (
        cpu_tdp <= cooler_tdp)

    return socket_ok and tdp_ok


def get_mm(value):

    return float(str(value).replace("mm", "").strip())


def check_cooler_cabinet(cooler, cabinet):

    cooler_height = get_mm(cooler["Cooler Height"])

    max_cooler_height = get_mm(cabinet["Max Cooler Height"])

    return cooler_height <= max_cooler_height




In [29]:
#new
def check_compatibility(build):

    results = {
        "CPU ↔ Motherboard": check_cpu_mobo(
            build["CPU"],
            build["Mother Board"]),

        "RAM ↔ Motherboard": check_ram_mobo(
            build["RAM"],
            build["Mother Board"]),

        "GPU ↔ PSU": check_gpu_psu(
            build["GPU"],
            build["PSU"]),

        "GPU ↔ Cabinet": check_gpu_cabinet(
            build["GPU"],
            build["Cabinet"]),

        "CPU ↔ Cooler": check_cpu_cooler(
            build["CPU"],
            build["Cooler"]),

        "Cooler ↔ Cabinet": check_cooler_cabinet(
            build["Cooler"],
            build["Cabinet"])
    }

    return results

#new
def is_build_compatible(compatibility_results):
    return all(compatibility_results.values())

In [30]:
def get_compatibility_summary(compatibility_results):

    issues = [
        component
        for component, result
        in compatibility_results.items()
        if not result]

    return {
        "is_compatible": len(issues) == 0,
        "issues": issues}

### **2. Model prediction**

In [31]:
def predict_components(user_input, model):

    prediction = model.predict(user_input)
    prediction = np.squeeze(prediction)
    return prediction

### **3. Encoded predictions ko actual IDs mein decode karna**

In [32]:
def decode_predictions(prediction):

    decoded_prediction = {}

    for col, encoded_value in zip(target_columns, prediction):

        decoded_value = target_encoders[col].inverse_transform([encoded_value])[0]
        decoded_prediction[col] = decoded_value
        
    return decoded_prediction

### **4. Actual component lookup**

In [33]:
def get_component(df, column_name, component_id):

    result = df[df[column_name] == component_id]

    if result.empty:
        raise ValueError(f"Component {component_id} not found in column {column_name}")

    return result.iloc[0]

In [34]:
def get_component_details(decoded_prediction):

    build = {}

    for component_name, df in component_dataframes.items():

        target_column = {
            "CPU": "cpu",
            "GPU": "gpu",
            "Mother Board": "mobo",
            "Cooler": "cpu_cooler",
            "PSU": "psu",
            "Storage": "storage",
            "RAM": "ram",
            "Cabinet": "cabinet",
            "Monitor": "monitor"
        }[component_name]

        component_id = decoded_prediction[target_column]

        id_column = components_id_col_name[
            component_name]

        build[component_name] = get_component(
            df,
            id_column,
            component_id)

    return build

### **5. Price calculation**

In [35]:
def calculate_build_price(build):

    price_columns = {
        "CPU": "price",
        "GPU": "gpu price",
        "Mother Board": "price",
        "Cooler": "Price",
        "PSU": "Approx Price (INR)",
        "Storage": "price",
        "RAM": "approx_price",
        "Cabinet": "Price",
        "Monitor": "Approx Price"
    }

    total = 0

    for component, item in build.items():

        price_column = price_columns[component]

        price = pd.to_numeric(
            item[price_column],
            errors="coerce"
        )

        if pd.notna(price):
            total += price

    return total

### **6. Final recommendation function**

In [36]:
def recommend_pc(
    budget,
    used_parts,
    usage_scenario,
    colour_theme,
    monitor_required,
    monitor_size,
    monitor_resolution,
    build_type,
    upgrade_path, model):

    # 1. Prepare user input
    user_input = preprocess_user_input(
        budget,
        used_parts,
        usage_scenario,
        colour_theme,
        monitor_required,
        monitor_size,
        monitor_resolution,
        build_type,
        upgrade_path)


    # 2. Predict component classes
    prediction = predict_components(user_input, model)


    # 3. Decode classes → component IDs
    decoded_prediction = decode_predictions(prediction)


    # 4. Component IDs → actual component rows
    build = get_component_details(decoded_prediction)


    # 6. Compatibility validation
    compatibility = check_compatibility(build)
    build_compatible = is_build_compatible(compatibility) #new

    summary = get_compatibility_summary(compatibility) #new

    # 5. Calculate total price
    total_price = calculate_build_price(build
)

    return {
        "components": build,
        "total_price": total_price,
        "compatibility": compatibility,
        "build_compatible": build_compatible, #new
        "issues": summary["issues"] #new
    }

### **Input Test**

In [37]:
result = recommend_pc(
    budget=40000,
    used_parts=0,
    usage_scenario="Budget Gaming + Office Usage",
    colour_theme="black",
    monitor_required=1,
    monitor_size=24,
    monitor_resolution="FHD",
    build_type="normal",
    upgrade_path="medium",
    model=Forest_model
)


In [38]:
result['components']

{'CPU': cpu_id                                          cpu106
 cpu_model                                Ryzen 5 5600G
 manufacturer                                       AMD
 socket_no                                          AM4
 compatible_socket         X570, X470, B550, B450, A520
 cores                                                6
 thread                                              12
 base_clock                                         3.9
 boost_clock                                        4.9
 chache_memory                                       19
 tdp_wattage                                         65
 ram_support                                       DDR4
 integrated_gpu                                     yes
 integrated_gpu_name                    Radeon Graphics
 cooler_recommended     Stock  Cooler, Tower Air Cooler
 cpu_tier                                             5
 price                                    ₹11,200 (New)
 Name: 105, dtype: object,
 'GPU': gpu id

In [39]:
joblib.dump(ohe, "models/ohe.pkl")

# Target encoders
joblib.dump(target_encoders, "models/target_encoders.pkl")

# Feature columns
joblib.dump(X.columns.tolist(), "models/feature_columns.pkl")

['models/feature_columns.pkl']